# Task B -- one change stacked on the 0.6410 recipe

This notebook runs four full-data arms. Every arm is byte-for-byte `b_reinit1_rdrop_full`, the Run 9 submission that scored **0.6410** on CodaBench, except for the single flag named in its row. Each arm writes its own CodaBench ZIP.

| setting | value |
|---|---|
| TAPT corpus | D0: Task B training comments + OffensEval Kannada, all 6,406 |
| TAPT text | `--val-frac 0 --min-words 1 --no-dedupe`, nothing held back |
| classifier | `--folds 1`, all 3,159 labelled rows per seed, no deduplication |
| reinitialization | **one** top encoder layer (`--reinit-layers 1`) |
| R-Drop | `--rdrop 0.5` |
| seeds | 42, 43, 44, 45, 46; probabilities averaged |
| score | CodaBench only; nothing is held out, so there is no local F1 |

## The four arms

| arm | the one flag added | question it answers | time |
|---|---|---|---|
| `s_epochs10` | `--epochs 10` | six epochs was tuned on Task A before R-Drop and TAPT existed. R-Drop makes each epoch teach less, because half the signal is now the agreement term between its two dropout passes. Does the model want longer? | 188 min |
| `s_control` | none | reruns Run 9 unchanged. Its CodaBench score against 0.6410 is a direct read on run-to-run noise, and it supplies the `test_probs.npy` the TF-IDF blend needs | 113 min |
| `s_tags` | `--tags` | appends gazetteer, mood and address tags to each comment. Measured as noise on the TF-IDF SVM at -0.007, but that model already has the gazetteer words as features; MuRIL sees `bommai` 19 times as wordpieces | 113 min |
| `s_nofgm` | `--no-fgm` | FGM is about 45% of runtime and has never been tested on Task B at all. If it ties, every future Task B run gets roughly twice as fast | 73 min |

A fifth output, `s_blend`, costs no GPU: it averages `s_control`'s probabilities with the TF-IDF SVM's at a fixed weight.

## Arms that are deliberately off

`--arms` can re-enable these, but the evidence already available says each will lose:

| arm | why it is off |
|---|---|
| `s_stopwords` | the 60-word frequency stoplist derived from this corpus contains `bjp`, `congress`, `dagar` and `desha`, which is the Political and Gender signal. 113 min to confirm a prediction we can make for free |
| `s_stem` | character normalization already measured -0.003 here and transliteration-lite +0.001. Wordpiece splits `madthare` into stem and suffixes already |
| `s_polarity` | the same shape as `b_abusive`, which scored 0.5718 against stock MuRIL's 0.5948. It also reads the external corpus's **labels**, which TAPT deliberately avoids, so it carries a rules question |
| `s_seeds10` | ten seeds instead of five. Run-to-run spread on this recipe is about 0.5 points, whatever 5 to 10 buys is smaller, and 1 point on the 395-row set is 2 rows |

Both `s_stopwords` and `s_stem` also change the classifier's input so it no longer matches the raw text TAPT adapted to, and that adaptation is worth +2.9 points. A loss there would be ambiguous between the idea and the mismatch.

## Before you start

About **8.9 hours** including one shared TAPT pass. A wall-clock guard stops any arm it cannot finish and rewrites `RESULTS.md` after each one, so an interrupted session still leaves usable output.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then **Save Version -> Save & Run All**. Never run this interactively: the session dies with the browser tab.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Run the four arms

`experiments/task_b/stack_full.py` builds the TAPT checkpoint once and shares it across every arm, then trains each arm and packages its ZIP. `--reference` sets the submission the agreement column is measured against; point it at Run 9's `predictions.csv` if you kept that file, because agreement against the older 0.5922 submission is much less informative.

Arms are ordered by value, and the guard skips from the bottom, so a short session loses `s_nofgm` rather than `s_epochs10`.

In [ ]:
import time
t0 = time.time()
run([sys.executable, "-u", "experiments/task_b/stack_full.py",
     "--budget-hours", "10.5",
     "--reserve-min", "20",
     "--arms", "s_epochs10", "s_control", "s_tags", "s_nofgm",
     "--reference", "submissions/b_tapt_5f/predictions.csv",
     "--out", "/kaggle/working"],
    log="artifacts/logs/stack_full.log")
print(f"\nelapsed {(time.time() - t0) / 3600:.2f} h")

## 2. Read the diagnostics

Neither column below is a score, and neither can be. Every arm trains on all 3,159 rows, so nothing is held out, and the official validation labels are never released.

`agree` is the share of the 395 predictions matching the reference submission. An arm agreeing on 97% of rows will land near it and is probably not worth a submission slot; one agreeing on 70% is a genuinely different bet.

`drift` is the total absolute gap between the arm's predicted class rates and the training prior, in points. A large number means the arm has collapsed onto a class, which is the one failure this notebook can catch without labels. The TF-IDF floor called `Violence` on 6 of 395 rows against a 7.0% training rate, and under macro-F1 that alone forfeits most of a class.

In [ ]:
RESULTS = pathlib.Path("/kaggle/working/RESULTS.md")
print(RESULTS.read_text() if RESULTS.exists() else "RESULTS.md not written -- check the log above")

subs = sorted(pathlib.Path("/kaggle/working/subs").glob("*.zip"))
print("\nZIPs ready to upload:")
for z in subs:
    with zipfile.ZipFile(z) as f:
        assert f.namelist() == ["predictions.csv"], (z.name, f.namelist())
    print(f"  {z.name:24s} {z.stat().st_size:6d} bytes")
assert subs, "no ZIPs were written -- every arm was skipped or failed"

## 3. Preserve the outputs

Download these individually rather than using Download All: the TAPT checkpoint in `artifacts/` is about a gigabyte.

`test_probs.npy` from `s_control` is the file worth keeping beyond this run. It is the unstacked recipe's probability matrix, and any future blend can be built from it without touching a GPU.

In [ ]:
OUT = pathlib.Path("/kaggle/working/stack_full_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for z in subs:
    shutil.copy2(z, OUT / z.name)
for tag in ["s_epochs10", "s_control", "s_tags", "s_nofgm", "s_blend"]:
    d = pathlib.Path("artifacts/runs") / tag
    for name in ["predictions.csv", "test_probs.npy"]:
        if (d / name).exists():
            shutil.copy2(d / name, OUT / f"{tag}_{name}")
for log in pathlib.Path("artifacts/logs").glob("s_*.log"):
    shutil.copy2(log, OUT / log.name)
if RESULTS.exists():
    shutil.copy2(RESULTS, OUT / "RESULTS.md")
print(sorted(p.name for p in OUT.iterdir()))

## 4. After CodaBench scores the submissions

Upload `s_control` first. Its score against Run 9's 0.6410 tells you how to read everything else: if a pure rerun of the identical recipe lands at 0.63 or 0.65, then a 1-point gap anywhere in this table means nothing, and only arms moving 2 points or more are worth acting on.

Then record every score in `docs/EXPERIMENTS.md` and `submissions/README.md`, including the arms that lost. A negative result that is written down is what stops the same idea being retried in three weeks.

Keep `b_reinit1_rdrop_full` as the candidate unless an arm beats 0.6410 by more than the noise `s_control` just measured.